# T1CViz Tutorial: Visualization for Neural Networks and Events

T1CViz provides interactive visualization tools for:
1. **Graph Visualization**: Render T1C-IR graphs as interactive HTML
2. **Event/Spike Visualization**: Process and display neuromorphic event data
3. **Pattern Detection**: Identify architectural patterns (ResNet, SPP, etc.)

## Prerequisites

```bash
pip install t1c-sdk
```

In [1]:
# Setup
import os
import numpy as np

# T1C packages (namespace package structure)
from t1c import ir   # Graph representation
from t1c import viz  # Visualization

# Create output directories
os.makedirs("viz", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Check available features
print(f"T1CViz version: {viz.__version__}")
print(f"Tonic available: {viz.TONIC_AVAILABLE}")
print(f"PIL available: {viz.PIL_AVAILABLE}")

T1CViz version: 0.0.1
Tonic available: False
PIL available: True


## 1. Graph Visualization

T1CViz renders T1C-IR graphs as interactive HTML with:
- Zoomable/pannable graph view
- Node details on hover
- Architecture pattern highlighting

In [2]:
# Create a simple feedforward SNN
nodes = {
    "input": ir.Input(np.array([784])),
    "fc1": ir.Affine(
        weight=np.random.randn(256, 784).astype(np.float32) * 0.01,
        bias=np.zeros(256, dtype=np.float32),
    ),
    "lif1": ir.LIF(
        tau=np.ones(256, dtype=np.float32) * 10.0,
        r=np.ones(256, dtype=np.float32),
        v_leak=np.zeros(256, dtype=np.float32),
        v_threshold=np.ones(256, dtype=np.float32),
    ),
    "fc2": ir.Affine(
        weight=np.random.randn(10, 256).astype(np.float32) * 0.01,
        bias=np.zeros(10, dtype=np.float32),
    ),
    "lif2": ir.LIF(
        tau=np.ones(10, dtype=np.float32) * 10.0,
        r=np.ones(10, dtype=np.float32),
        v_leak=np.zeros(10, dtype=np.float32),
        v_threshold=np.ones(10, dtype=np.float32),
    ),
    "output": ir.Output(np.array([10])),
}

edges = [
    ("input", "fc1"),
    ("fc1", "lif1"),
    ("lif1", "fc2"),
    ("fc2", "lif2"),
    ("lif2", "output"),
]

graph = ir.Graph(nodes=nodes, edges=edges)
print(f"Created graph: {len(graph.nodes)} nodes, {len(graph.edges)} edges")

Created graph: 6 nodes, 5 edges


In [3]:
# Export to interactive HTML
viz.export_html(graph, "viz/snn_visualization.html", title="Simple SNN")
print("Exported to viz/snn_visualization.html")
print("Open this file in a browser to explore the graph interactively.")

Exported to viz/snn_visualization.html
Open this file in a browser to explore the graph interactively.


In [4]:
# Serialize graph for visualization (see what data is included)
data = viz.graph_to_dict(graph)

print(f"Serialized data keys: {list(data.keys())}")
print(f"\nSummary:")
print(f"  Node count: {data['summary']['node_count']}")
print(f"  Edge count: {data['summary']['edge_count']}")
print(f"  Total params: {data['summary']['total_params']:,}")

Serialized data keys: ['version', 'nodes', 'edges', 'metadata', 'summary', 'groups', 'node_to_group']

Summary:
  Node count: 6
  Edge count: 5
  Total params: 204,594


## 2. Event/Spike Visualization

T1CViz processes neuromorphic event data into:
- **Frames**: Accumulated event counts per time bin
- **Rasters**: Spike times per neuron
- **Grids**: Spatial bins for larger sensors

In [5]:
# Generate synthetic event data
# Events are structured arrays with fields: x, y, t, p (polarity)
n_events = 5000
sensor_size = (28, 28)

events = np.zeros(n_events, dtype=[
    ('x', np.int32),
    ('y', np.int32),
    ('t', np.float64),  # microseconds
    ('p', np.int32),    # polarity: -1 or 1
])

# Random event locations and times
events['x'] = np.random.randint(0, sensor_size[0], n_events)
events['y'] = np.random.randint(0, sensor_size[1], n_events)
events['t'] = np.sort(np.random.uniform(0, 100000, n_events))  # 100ms
events['p'] = np.random.choice([-1, 1], n_events)

print(f"Generated {n_events} events")
print(f"Time range: {events['t'].min():.0f} - {events['t'].max():.0f} us")
print(f"Sensor size: {sensor_size}")

Generated 5000 events
Time range: 44 - 99985 us
Sensor size: (28, 28)


In [6]:
# Convert events to frames (accumulated counts per time bin)
frames = viz.events_to_frames(
    events,
    sensor_size=sensor_size,
    n_frames=25,
)

print(f"Frame shape: {frames.shape}")
print(f"  (n_frames, channels, height, width)")
print(f"  Channels: 2 (ON/OFF polarity)")

Frame shape: (25, 2, 28, 28)
  (n_frames, channels, height, width)
  Channels: 2 (ON/OFF polarity)


In [7]:
# Convert events to raster (spike times per neuron)
# n_neurons = width * height for 2D sensors
n_neurons = sensor_size[0] * sensor_size[1]

raster = viz.events_to_raster(
    events,
    n_neurons=n_neurons,
    time_bins=100,
)

print(f"Raster shape: {raster.shape}")
print(f"  (time_bins, n_neurons)")
print(f"  n_neurons = {sensor_size[0]} * {sensor_size[1]} = {n_neurons}")

Raster shape: (100, 784)
  (time_bins, n_neurons)
  n_neurons = 28 * 28 = 784


In [8]:
# Convert events to grid (spatial bins)
grid = viz.events_to_grid(
    events,
    sensor_size=sensor_size,
    n_bins=5,  # 5 time bins
)

print(f"Grid shape: {grid.shape}")
print(f"  (n_bins, height, width)")

Grid shape: (5, 28, 28, 3)
  (n_bins, height, width)


In [9]:
# Export event visualization to HTML
viz.export_events_html(
    events,
    "viz/tutorial_events.html",
    sensor_size=sensor_size,
    title="Tutorial: Synthetic Events",
    n_frames=30,
)

print("Exported to viz/tutorial_events.html")
print("Features:")
print("  - Animated frame playback")
print("  - Spike raster plot")
print("  - Event statistics")

Exported to viz/tutorial_events.html
Features:
  - Animated frame playback
  - Spike raster plot
  - Event statistics


## 3. Pattern Detection

T1CViz can detect common architectural patterns in graphs:
- **SkipConnection**: ResNet-style residual blocks
- **SPP**: Spatial Pyramid Pooling
- **SPPF**: Fast SPP (YOLOv5+)
- **RepConv**: Reparameterizable convolutions

In [10]:
# Create a graph with skip connections (ResNet-style)
skip_nodes = {
    "input": ir.Input(np.array([64, 14, 14])),
    
    # Processing path
    "conv1": ir.Conv2d(
        weight=np.random.randn(64, 64, 3, 3).astype(np.float32) * 0.01,
        bias=np.zeros(64, dtype=np.float32),
        stride=(1, 1), padding=(1, 1),
    ),
    "lif1": ir.LIF(
        tau=np.ones(64, dtype=np.float32) * 10.0,
        r=np.ones(64, dtype=np.float32),
        v_leak=np.zeros(64, dtype=np.float32),
        v_threshold=np.ones(64, dtype=np.float32),
    ),
    "conv2": ir.Conv2d(
        weight=np.random.randn(64, 64, 3, 3).astype(np.float32) * 0.01,
        bias=np.zeros(64, dtype=np.float32),
        stride=(1, 1), padding=(1, 1),
    ),
    
    # Skip connection (residual add)
    "skip": ir.Skip(
        input_type={"input": np.array([64, 14, 14])},
        skip_type="residual",
    ),
    
    "lif2": ir.LIF(
        tau=np.ones(64, dtype=np.float32) * 10.0,
        r=np.ones(64, dtype=np.float32),
        v_leak=np.zeros(64, dtype=np.float32),
        v_threshold=np.ones(64, dtype=np.float32),
    ),
    
    "output": ir.Output(np.array([64, 14, 14])),
}

skip_edges = [
    ("input", "conv1"),     # Main path
    ("conv1", "lif1"),
    ("lif1", "conv2"),
    ("conv2", "skip"),      # Processed output to skip
    ("input", "skip"),      # Direct skip connection
    ("skip", "lif2"),       # Merged output
    ("lif2", "output"),
]

skip_graph = ir.Graph(nodes=skip_nodes, edges=skip_edges)
print(f"Created ResNet-style graph: {len(skip_graph.nodes)} nodes")

Created ResNet-style graph: 7 nodes


In [11]:
# Detect patterns
patterns = viz.detect_all_patterns(skip_graph)

print(f"Patterns detected: {len(patterns)}")
for pattern in patterns:
    print(f"  - {pattern.pattern_type}: {pattern.group_id}")
    print(f"    Nodes: {pattern.nodes}")

Patterns detected: 1
  - SkipConnection: skip_input_skip
    Nodes: ['skip', 'conv1', 'lif1', 'conv2']


In [12]:
# Export with pattern highlighting
viz.export_html(
    skip_graph, 
    "viz/resnet_block.html", 
    title="ResNet Block with Skip Connection"
)
print("Exported to viz/resnet_block.html")
print("The skip connection pattern will be highlighted in the visualization.")

Exported to viz/resnet_block.html
The skip connection pattern will be highlighted in the visualization.


## 4. Performance Considerations

T1CViz provides optimized implementations for event processing:

| Implementation | Speed | Requirements |
|----------------|-------|-------------|
| NumPy (default) | ~23M events/sec | None |
| Tonic/Numba | ~30M events/sec | `pip install tonic` |

In [13]:
# Check which implementation is being used
print(f"Tonic available: {viz.TONIC_AVAILABLE}")

if viz.TONIC_AVAILABLE:
    print("Using Tonic/Numba implementation (faster)")
else:
    print("Using NumPy implementation")
    print("Install tonic for ~30% speedup: pip install tonic")

Tonic available: False
Using NumPy implementation
Install tonic for ~30% speedup: pip install tonic


In [14]:
# Force NumPy implementation (useful for debugging)
frames_numpy = viz.events_to_frames(
    events,
    sensor_size=sensor_size,
    n_frames=25,
    use_tonic=False,  # Force NumPy
)
print(f"NumPy frames shape: {frames_numpy.shape}")

NumPy frames shape: (25, 2, 28, 28)


## Summary

T1CViz capabilities:

| Function | Purpose |
|----------|---------|  
| `viz.export_html()` | Interactive graph visualization |
| `viz.graph_to_dict()` | Serialize graph for visualization |
| `viz.events_to_frames()` | Convert events to frame arrays |
| `viz.events_to_raster()` | Convert events to spike rasters |
| `viz.events_to_grid()` | Convert events to spatial grids |
| `viz.export_events_html()` | Interactive spike visualization |
| `viz.detect_all_patterns()` | Find architectural patterns |